# Post-infection immunity

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/06-immunity.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Assumptions about immunity shape the long-term dynamics of infectious diseases.
Even simple models admit several common structural choices. This notebook
compares SI, SIR, SIS, and SIRS (and a two-susceptible variant of SIRS).

## Structural assumptions

![](figures/06/immunity_structures.svg)


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import Compartments, Param, Property, PropertyMap, SavePlan, SaveRequest
from summer4.epi import EpiModel

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS = {"index": "time", "value": "proportion"}


def infectious_series(res: Any, state: Property) -> pd.Series:
    return res["comp"].select(state["infectious"]).to_pandas().iloc[:, 0]


def state_frame(res: Any) -> pd.DataFrame:
    frame = res["comp"].to_pandas()
    frame.columns = [
        c.split("state=")[-1].split("_pop=")[0] for c in frame.columns
    ]
    return frame


In [ ]:
def build_immunity_model(
    disease_states: tuple[str, ...],
    *,
    extra_infection_sources: tuple[str, ...] = (),
) -> tuple[Any, PropertyMap, Property]:
    """Frequency-dependent infection from susceptible (+ optional extra sources)."""
    state = Property("state", disease_states)
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    model = EpiModel(pmap, infectious=state["infectious"])
    model.set_mixing_matrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_infection_frequency_flow(
        "infection",
        state["susceptible"],
        state["infectious"],
        Param("contact_rate"),
    )
    for src in extra_infection_sources:
        model.add_infection_frequency_flow(
            f"infection_from_{src}",
            state[src],
            state["infectious"],
            Param("contact_rate"),
        )
    return model, pmap, state


model_config = {"population": 1.0, "seed": 0.001, "end_time": 40.0}
parameters = {
    "contact_rate": 1.0,
    "recovery_rate": 0.333,
    "immunity_waning": 0.1,
}
times = np.linspace(0.0, model_config["end_time"], 401)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)


def seed_y0(pmap: PropertyMap, state: Property) -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = model_config["population"] - model_config["seed"]
    y0[pmap.select(state["infectious"])] = model_config["seed"]
    return y0


def run_model(model: EpiModel, pmap: PropertyMap, state: Property, params: dict[str, float]) -> Any:
    return model.compile().run(
        params,
        seed_y0(pmap, state),
        t0=0.0,
        t1=model_config["end_time"],
        dt=0.1,
        save=plan,
        solver="dopri5",
    )


### SI structure

Permanent infection and infectiousness: anyone who enters `infectious` stays
there. Few pathogens are well represented by this — permanent infection does
not usually imply permanent infectiousness.


In [ ]:
si_model, si_pmap, si_state = build_immunity_model(("susceptible", "infectious"))
si_res = run_model(si_model, si_pmap, si_state, parameters)
si_values = state_frame(si_res)
si_i = infectious_series(si_res, si_state)
si_values.plot(labels=AXIS, title="SI")
assert float(si_i.iloc[-1]) > float(si_i.iloc[0]), "SI infectiousness should accumulate"


### SIR structure

Recovery confers permanent, complete immunity to reinfection — appropriate for
some infections.


In [ ]:
sir_model, sir_pmap, sir_state = build_immunity_model(
    ("susceptible", "infectious", "recovered")
)
sir_model.add_transition_flow(
    "recovery", sir_state["infectious"], sir_state["recovered"], Param("recovery_rate")
)
sir_res = run_model(sir_model, sir_pmap, sir_state, parameters)
sir_values = state_frame(sir_res)
sir_i = infectious_series(sir_res, sir_state)
sir_values.plot(labels=AXIS, title="SIR")
assert float(sir_i.max()) > model_config["seed"]
assert float(sir_i.iloc[-1]) < float(sir_i.max()), "SIR epidemic should decline after the peak"


### SIS structure

No immunity: recovery returns people to the susceptible pool at the same risk
as never-infected individuals.


In [ ]:
sis_model, sis_pmap, sis_state = build_immunity_model(("susceptible", "infectious"))
sis_model.add_transition_flow(
    "recovery", sis_state["infectious"], sis_state["susceptible"], Param("recovery_rate")
)
sis_res = run_model(sis_model, sis_pmap, sis_state, parameters)
sis_values = state_frame(sis_res)
sis_i = infectious_series(sis_res, sis_state)
sis_values.plot(labels=AXIS, title="SIS")
assert float(sis_i.iloc[-1]) > 0.05, "SIS should retain endemic infectiousness"


### SIRS structure

Immunity after recovery lasts only a limited time. After an initial wave
depletes susceptibles, the model approaches an equilibrium where new infections
offset waning of immunity.


In [ ]:
sirs_model, sirs_pmap, sirs_state = build_immunity_model(
    ("susceptible", "infectious", "recovered")
)
sirs_model.add_transition_flow(
    "recovery", sirs_state["infectious"], sirs_state["recovered"], Param("recovery_rate")
)
sirs_model.add_transition_flow(
    "immunity_waning",
    sirs_state["recovered"],
    sirs_state["susceptible"],
    Param("immunity_waning"),
)
sirs_res = run_model(sirs_model, sirs_pmap, sirs_state, parameters)
sirs_values = state_frame(sirs_res)
sirs_i = infectious_series(sirs_res, sirs_state)
sirs_values.plot(labels=AXIS, title="SIRS")
assert float(sirs_i.iloc[-1]) > float(sir_i.iloc[-1]), (
    "waning immunity should leave more late infectious prevalence than permanent SIR immunity"
)


## Tracking immunity status

Compartmental models are memoryless: the future depends only on the current
state. Flow rates at a time do not tell you the history of who just arrived in
a destination compartment. Extra compartments can track that history. Consider
an SIRS variant with a second susceptible compartment for people whose immunity
has waned:

![](figures/06/sirs2_structure.svg)


In [ ]:
sirs2_model, sirs2_pmap, sirs2_state = build_immunity_model(
    ("susceptible", "infectious", "recovered", "susceptible_2"),
    extra_infection_sources=("susceptible_2",),
)
sirs2_model.add_transition_flow(
    "recovery", sirs2_state["infectious"], sirs2_state["recovered"], Param("recovery_rate")
)
sirs2_model.add_transition_flow(
    "immunity_waning",
    sirs2_state["recovered"],
    sirs2_state["susceptible_2"],
    Param("immunity_waning"),
)
sirs2_res = run_model(sirs2_model, sirs2_pmap, sirs2_state, parameters)
sirs2_values = state_frame(sirs2_res)
sirs2_i = infectious_series(sirs2_res, sirs2_state)
sirs2_values.plot(labels=AXIS, title="SIRS with tracked reinfection susceptibles")

# Dynamics of infectiousness match SIRS; the split susceptibles are the bookkeeping gain.
np.testing.assert_allclose(sirs2_i.to_numpy(), sirs_i.to_numpy(), rtol=1e-3, atol=1e-3)
assert float(sirs2_values["susceptible_2"].iloc[-1]) > 0.0


The infectious trajectory matches the simpler SIRS model. The difference is
two susceptible compartments, so outputs can separate never-infected people from
those whose immunity has waned (and attribute infections to reinfection) —
quantities unavailable from the single-susceptible SIRS.

## Comparison


In [ ]:
compare = pd.DataFrame(
    {
        "si": si_i,
        "sir": sir_i,
        "sis": sis_i,
        "sirs": sirs_i,
        "sirs2": sirs2_i,
    }
)
compare.plot(labels=AXIS, title="Infectious prevalence under immunity assumptions")
assert float(compare["si"].iloc[-1]) > float(compare["sir"].iloc[-1])
assert float(compare["sis"].iloc[-1]) > float(compare["sir"].iloc[-1])
